In [4]:
import pandas as pd
import os
import zipfile
from pathlib import Path
import numpy as np
import re
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from functools import reduce
import gc

In [5]:
def dropna_with_counts(df, column_order):
    cleaned = df.copy()
    records = []

    # Check the selected columns first, then all remaining columns.
    # This preserves the final sample produced by your original dropna().
    columns = column_order + [col for col in cleaned.columns if col not in column_order]

    for col in columns:
        before = len(cleaned)
        dropped = int(cleaned[col].isna().sum())
        cleaned = cleaned.dropna(subset=[col])

        records.append({
            "Column checked": col,
            "Rows before": before,
            "Rows dropped": dropped,
            "Rows remaining": len(cleaned)
        })

    return cleaned.reset_index(drop=True), pd.DataFrame(records)

* Variables and Functions setup

In [6]:
zip_folder = 'raw_data'
data_folder = 'extracted_data'
survey_year_79 = [y for y in range(1979,2023) if y not in [1995, 1997, 1999, 2001, 2003, 2005, 2007, 2009, 2011, 2013, 2015, 2017, 2019, 2021]]
survey_year_97 = [y for y in range(1997,2024) if y not in [2012,2014,2016,2018,2020,2022]]
race_mapping = {
    3: 'White', 5: 'White', 6: 'White', 7: 'White', 11: 'White', 12: 'White', 
    22: 'White', 23: 'White', 24: 'White', 25: 'White', 27: 'White', 29: 'White', 
    1: 'Black',
    15: 'Hispanic', 16: 'Hispanic', 17: 'Hispanic', 18: 'Hispanic', 19: 'Hispanic', 
    20: 'Hispanic', 21: 'Hispanic',
    2: 'Asian/PI', 4: 'Asian/PI', 8: 'Asian/PI', 10: 'Asian/PI', 13: 'Asian/PI', 
    14: 'Asian/PI', 26: 'Asian/PI',
    0: 'Native/Other', 9: 'Native/Other', 28: 'Native/Other' 
}
sex_mapping = {1:"Male", 2:"Female"}
race_map_97 = {1: 'Black', 2:'Hispanic', 3:'Mixed Race', 4:'Non-black/Non-Hispanic'}

In [8]:
def create_rnum_dict(file_path):
    rnum_to_var = {}
    pattern = re.compile(r"^([A-Za-z0-9]+\.\d{2}[A-Za-z0-9]*)\s+\[(.*?)\]\s+Survey Year:\s*(.*)")
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
        for line in file:
            match = pattern.search(line.strip())
            if match:
                rnum = re.sub(r'[^a-zA-Z0-9]', '', match.group(1))
                var_name = match.group(2)
                survey_year_79 = match.group(3).strip()
                if re.match(r"^\d{4}$", survey_year_79):
                    var_name = f"{var_name}_{survey_year_79}"
                rnum_to_var[rnum] = var_name     
    return rnum_to_var

# 79 rename dictionary
def rename_columns(col):
    if 'EMPLOYERS_ALL_' in col:
        clean_name = col.replace('EMPLOYERS_ALL_', '')
        if 'UID' in col:
            return clean_name
        match_gap = re.match(r'(.+)_(\d{4})\.(\d+)\.(\d+)$',clean_name)
        if match_gap:
            var, year, job_num, gap_num = match_gap.groups()
            return f"{var}.{job_num}_{gap_num}_{year}"
        match_job = re.match(r'(.+)_(\d{4})\.(\d+)$',clean_name)
        if match_job:
            var, year, job_num = match_job.groups()
            return f"{var}.{job_num}_{year}"
        return clean_name
    elif 'CASEID' in col:
        return 'CASEID'
    elif 'HGCREV' in col:
        parts = col.split('_')
        return f"{parts[0][:-2]}_{parts[1]}"
    elif 'FAM-30_1_1979' in col:
        return 'race'
    elif 'SAMPLE_SEX_1979' in col:
        return 'sex'
    return col

* Occupation major grouping maping

In [9]:
occ70_dict = {}
for x in range(0, 196):
    occ70_dict[x] = "Professional"
for x in range(201, 246):
    occ70_dict[x] = "Managers"
for x in range(260,281):
    occ70_dict[x] = "Sales"
for x in range(301,396):
    occ70_dict[x] = "Clerical"
for x in range(401,581):
    occ70_dict[x] = "Craftsmen"
# for x in range(601,727):
#     occ70_dict[x] = "Operatives"
# for x in range(740,786):
#     occ70_dict[x] = "Laborers"
for x in range(601,786):
    occ70_dict[x] = "Operatives and Laborers"
for x in range(901,985):
    occ70_dict[x] = "Service"

    
occ00_dict = {}
# Professional
for x in [range(62,71),80,82,93,range(100,352),353,354,361,362,363,462,493,601,903,904]:
    if type(x) is int:
        occ00_dict[x] = "Professional"
    elif type(x) is range:
        for n in x:
            occ00_dict[n] = "Professional"
# Managers
for x in [range(0,54),56,72,73,83,range(84,92),94,95,931]:
    if type(x) is int:
        occ00_dict[x] = "Managers"
    elif type(x) is range:
        for n in x:
            occ00_dict[n] = "Managers"
# Sales
for x in [range(475,493),range(494,497)]:
    if type(x) is int:
        occ00_dict[x] = "Sales"
    elif type(x) is range:
        for n in x:
            occ00_dict[n] = "Sales"
# Clerical
for x in [54,60,81,472,474,500,range(510,556),range(556,594)]:
    if type(x) is int:
        occ00_dict[x] = "Clerical"
    elif type(x) is range:
        for n in x:
            occ00_dict[n] = "Clerical"
# Craftsmen
for x in [352,441,range(621,626),range(630,637),range(642,654),range(666,672),674,676,range(700,761),range(762,771),774,780,803,806,813,816,range(823,826),
          833,835,844,845,range(850,853),range(860,864),875,876,891,900,920,941,range(951,957)]:
    if type(x) is int:
        occ00_dict[x] = "Craftsmen"
    elif type(x) is range:
        for n in x:
            occ00_dict[n] = "Craftsmen"
# # Operatives
# for x in [range(501,504),640,range(680,692),694,range(771,774),775,range(781,803),804,range(810,813),814,815,range(820,823),range(826,833),834,range(836,843),846,range(853,856),
#           range(864,875),range(880,891),range(892,895),896,range(911,916),923,933,935,936,942,950,960,964,965,973]:
#     if type(x) is int:
#         occ00_dict[x] = "Operatives"
#     elif type(x) is range:
#         for n in x:
#             occ00_dict[n] = "Operatives"
# # Laborers
# for x in [421,425,434,435,600,range(602,614),626,660,672,673,675,692,693,761,895,range(961,964),972,974,975]:
#     if type(x) is int:
#         occ00_dict[x] = "Laborers"
#     elif type(x) is range:
#         for n in x:
#             occ00_dict[n] = "Laborers"
# Operatives and Laborers
for x in [range(501,504),640,range(680,692),694,range(771,774),775,range(781,803),804,range(810,813),814,815,range(820,823),range(826,833),834,range(836,843),846,range(853,856),
          range(864,875),range(880,891),range(892,895),896,range(911,916),923,933,935,936,942,950,960,964,965,973,421,425,434,435,600,range(602,614),626,660,672,673,675,692
          ,693,761,895,range(961,964),972,974,975]:
    if type(x) is int:
        occ00_dict[x] = "Operatives and Laborers"
    elif type(x) is range:
        for n in x:
            occ00_dict[n] = "Operatives and Laborers"
# Service
for x in [360,364,365,range(370,421),range(422,425),430,432,range(442,462),464,465,934]:
    if type(x) is int:
        occ00_dict[x] = "Service"
    elif type(x) is range:
        for n in x:
            occ00_dict[n] = "Service"

* Extracting files from zip files in raw_data folder, using zip_extract.py
  

In [10]:
%run zip_extract.py

* Generating csv map

In [11]:
# Create diction of CPI, base year as 2020
df_cpi = pd.read_csv('cpi.csv')
df_cpi['observation_date'] = df_cpi['observation_date'].str[:4].astype(int)
base = df_cpi.loc[df_cpi['observation_date'] == 2023, 'CPIAUCSL'].values[0]
df_cpi['ratio'] = df_cpi['CPIAUCSL'] / base
cpi_map = df_cpi.set_index('observation_date')['ratio'].to_dict()

### Experience

In [12]:
df79_exp = pd.read_csv(os.path.join(data_folder, '79-experience.csv'))
rename_dict = create_rnum_dict(os.path.join(data_folder, '79-experience.cdb'))
df79_exp = df79_exp.rename(columns = rename_dict)
df79_exp = df79_exp.rename(columns = rename_columns)
df79_exp = df79_exp.replace([-1, -2, -3, -4,-5], np.nan)

# Start Week
start_cols = [col for col in df79_exp.columns if col.startswith("STARTWEEK.")]
df79_startweek_long = df79_exp[["CASEID"] + start_cols].melt(id_vars="CASEID",value_vars=start_cols,var_name="variable",value_name="Startweek")
df79_startweek_long.dropna(subset=['Startweek'],inplace=True)
df79_startweek_long[["job_num","year"]] = df79_startweek_long["variable"].str.extract(r"STARTWEEK\.(\d+)_(\d{4})")
df79_startweek_long["year"] = pd.to_numeric(df79_startweek_long["year"])
df79_startweek_long["job_num"] = pd.to_numeric(df79_startweek_long["job_num"])
df79_startweek_long = df79_startweek_long.drop(columns="variable")

# Stop Week
stop_cols = [col for col in df79_exp.columns if col.startswith("STOPWEEK.")]
df79_stopweek_long = df79_exp[["CASEID"] + stop_cols].melt(id_vars="CASEID",value_vars=stop_cols,var_name="variable",value_name="Stopweek")
df79_stopweek_long.dropna(subset=['Stopweek'],inplace=True)
df79_stopweek_long[["job_num","year"]] = df79_stopweek_long["variable"].str.extract(r"STOPWEEK\.(\d+)_(\d{4})")
df79_stopweek_long["year"] = pd.to_numeric(df79_stopweek_long["year"])
df79_stopweek_long["job_num"] = pd.to_numeric(df79_stopweek_long["job_num"])
df79_stopweek_long = df79_stopweek_long.drop(columns="variable")

# Merging
df79_jobweeks_long = pd.merge(df79_startweek_long,df79_stopweek_long,on=["CASEID", "year", "job_num"],how="inner")
df79_jobweeks_long = df79_jobweeks_long.sort_values(["CASEID", "year", "job_num"]).reset_index(drop=True)
df79_jobweeks_long = df79_jobweeks_long[df79_jobweeks_long['Stopweek'] >= df79_jobweeks_long['Startweek']].copy()
df79_jobweeks_long['segment'] = df79_jobweeks_long['Stopweek'] - df79_jobweeks_long['Startweek']

# Minus Within Job Gap
df79_gap = pd.read_csv(os.path.join(data_folder, '79-job_gap.csv'))
rename_dict = create_rnum_dict(os.path.join(data_folder, '79-job_gap.cdb'))
df79_gap = df79_gap.rename(columns = rename_dict)
df79_gap = df79_gap.rename(columns = rename_columns)
df79_gap = df79_gap.replace([-1, -2, -3, -4,-5], np.nan)

gap_cols = [c for c in df79_gap.columns if c.startswith(("PERSTAR",'PERSTOP'))]
df79_gap_long = df79_gap[['CASEID']+gap_cols].melt(id_vars = 'CASEID', value_vars = gap_cols, var_name = 'variable')
df79_gap_long.dropna(subset=['value'],inplace=True)
df79_gap_long[["gap_var", "job_num", "gap_num", "year"]] = df79_gap_long["variable"].str.extract(r"(PERSTAR|PERSTOP).(\d+)_(\d+)_(\d{4})")
df79_gap_long["job_num"] = pd.to_numeric(df79_gap_long["job_num"])
df79_gap_long["gap_num"] = pd.to_numeric(df79_gap_long["gap_num"])
df79_gap_long["year"] = pd.to_numeric(df79_gap_long["year"])
df79_gap_long = df79_gap_long.drop(columns="variable")
df79_gap_long = df79_gap_long.pivot_table(index=["CASEID", "year", "job_num", "gap_num"],columns="gap_var",values="value",aggfunc="first").reset_index()
df79_gap_long['duration'] = df79_gap_long['PERSTOP'] - df79_gap_long['PERSTAR']
df79_gap_sum = df79_gap_long.groupby(['CASEID','year','job_num'],as_index=False)['duration'].sum()

# Merging with exp df and Generating tenure
df79_exp = df79_jobweeks_long.merge(df79_gap_sum,on=['CASEID','year','job_num'],how='left')
df79_exp['duration'] = df79_exp['duration'].fillna(0)
df79_exp['segment_new'] = df79_exp['segment'] - df79_exp['duration']
# df79_exp = df79_exp.drop(columns='duration')

df79_exp = df79_exp[["CASEID", "job_num", "year", "Startweek", "segment"]].copy()
df79_exp = df79_exp.sort_values(["CASEID", "year", "Startweek", "job_num"])
df79_exp["tenure"] = df79_exp.groupby(["CASEID","job_num"])["segment"].cumsum()
df79_exp = df79_exp[["CASEID", "job_num", "year", "tenure"]].reset_index(drop=True)
df79_exp['tenure'] = df79_exp['tenure']

# Save memory spaces
del df79_startweek_long, df79_stopweek_long, df79_jobweeks_long, df79_gap_long, df79_gap_sum
gc.collect()

27

### NLSY79

In [13]:
df79 = pd.read_csv(os.path.join(data_folder, 'nlsy79.csv'))
rename_dict = create_rnum_dict(os.path.join(data_folder, 'nlsy79.cdb'))
df79 = df79.rename(columns = rename_dict)
df79 = df79.rename(columns=rename_columns)
df79 = df79.replace([-1, -2, -3, -4,-5], np.nan)

hour_cols = [col for col in df79.columns if col.startswith('HOURSWEEK.')]
wage_cols = [col for col in df79.columns if col.startswith('HRLY_WAGE.')]
occ_cols = [col for col in df79.columns if col.startswith('OCC.')]
asvab_cols = [c for c in df79.columns if c.startswith("ASVAB")]

for c in occ_cols:
    original = df79[c].copy()
    if int(c[-4:]) < 2002:
        df79[c] = original.map(occ70_dict)
    elif int(c[-4:]) == 2002:
        df79[c] = original.map(occ00_dict)
    elif int(c[-4:]) > 2002:
        df79[c] = (original / 10).map(occ00_dict)

for c in wage_cols:
    year = int(c[-4:])
    df79[c] = (df79[c] / 100)/ cpi_map[year]

In [14]:
# # ASVAB three month or one year selection
# asvab_cols = [c for c in df79.columns if c.startswith("ASVAB")]

# # Comparing between 3 month grouping and one year grouping
# df_group = df79.copy()
# df_group["group_3month"] = (df_group["Q1-3_A~Y_1979"].astype(str) + "_Q" + (((df_group["Q1-3_A~M_1979"] - 1) // 3) + 1).astype(str))

# mean1 = (df_group.groupby("group_3month")[asvab_cols].mean().sort_index())
# mean2 = (df_group.groupby("Q1-3_A~Y_1979")[asvab_cols].mean().sort_index())

# ax = mean1.plot(figsize=(12, 6),linewidth=1)

# year_positions = {}
# for position, group_label in enumerate(mean1.index):
#     year = str(group_label).split("_")[0]
#     year_positions.setdefault(year, []).append(position)

# year_positions = {year: np.mean(positions) for year, positions in year_positions.items()}

# for column, line in zip(asvab_cols, ax.lines):
#     x_dots = []
#     y_dots = []
#     for year, value in mean2[column].items():
#         year_key = str(int(year))
#         if year_key in year_positions and pd.notna(value):
#             x_dots.append(year_positions[year_key])
#             y_dots.append(value)
#     ax.scatter(x_dots, y_dots, color=line.get_color(), edgecolor="black", linewidth=0.5,
#         s=45, zorder=5, label="_nolegend_")
# q1_positions = [i for i, label in enumerate(mean1.index) if str(label).upper().endswith("_Q1")]
# q1_labels = [str(mean1.index[i]).upper() for i in q1_positions]
# ax.set_xlabel("")
# ax.set_xticks(q1_positions)
# ax.set_xticklabels(q1_labels, rotation=0)
# ax.set_ylabel("Mean ASVAB score")
# ax.set_title("Three-Month ASVAB Means compared with One-Year Means - NLSY79 cohort")
# ax.get_legend().remove()
# ax.grid(alpha=0.3)

# plt.tight_layout()
# plt.show()

In [15]:
# Normalization of ASVAB scores using Three-Year age group
df79 = pd.concat([df79,(df79["Q1-3_A~Y_1979"].astype(str)+ "_Q"+ (((df79["Q1-3_A~M_1979"] - 1) // 3) + 1).astype(str)).rename("group_3month")], axis=1)
std_cols = [f"{col}_std" for col in asvab_cols]
df79[std_cols] = df79.groupby("group_3month")[asvab_cols].transform(lambda x: (x - x.mean()) / x.std(ddof=1))
# df_79ability[std_cols].plot.density()

# Applying PCA analysis
valid_r = df79[std_cols].notna().all(axis=1)
valid_df = df79.loc[valid_r, std_cols]
pca = PCA(n_components=2)
pca_scores = pca.fit_transform(valid_df)
print("Variance explained by PC1:", f"{pca.explained_variance_ratio_[0]:.3f}")
df79["ASVAB"] = np.nan
df79.loc[valid_r, "ASVAB"] = pca_scores[:, 0]
# df79["ASVAB"] = df79["ASVAB"].rank(pct=True)
# df79["ASVAB"] = (df79["ASVAB"] - df79["ASVAB"].mean()) / df79["ASVAB"].std(ddof=1)
df79.drop(columns=asvab_cols+std_cols+["group_3month"], inplace=True)
df79.dropna(subset=['ASVAB'], inplace=True)

Variance explained by PC1: 0.686


In [16]:
# Hoursweek
df79_hrsweek_long = df79.melt(id_vars=["CASEID"],value_vars=hour_cols,var_name="variable",value_name="HOURSWEEK").dropna(subset=["HOURSWEEK"])
df79_hrsweek_long[["job_num", "year"]] = df79_hrsweek_long["variable"].str.extract(r"HOURSWEEK\.(\d+)_(\d{4})")
df79_hrsweek_long["job_num"] = df79_hrsweek_long["job_num"].astype(int)
df79_hrsweek_long["year"] = df79_hrsweek_long["year"].astype(int)
df79_hrsweek_long = df79_hrsweek_long.drop(columns=["variable"]).sort_values(["CASEID", "year", "job_num"]).reset_index(drop=True)

# Wage
df79_wage_long = df79.melt(id_vars=["CASEID"],value_vars=wage_cols,var_name="variable",value_name="WAGE").dropna(subset=["WAGE"])
df79_wage_long[["job_num", "year"]] = df79_wage_long["variable"].str.extract(r"HRLY_WAGE\.(\d+)_(\d{4})")
df79_wage_long["job_num"] = df79_wage_long["job_num"].astype(int)
df79_wage_long["year"] = df79_wage_long["year"].astype(int)
df79_wage_long = df79_wage_long.drop(columns=["variable"]).sort_values(["CASEID", "year", "job_num"]).reset_index(drop=True)

# Occupation
df79_occ_long = df79.melt(id_vars=["CASEID"],value_vars=occ_cols,var_name="variable",value_name="OCC").dropna(subset=["OCC"])
df79_occ_long[["job_num", "year"]] = df79_occ_long["variable"].str.extract(r"OCC\.(\d+)_(\d{4})")
df79_occ_long["job_num"] = df79_occ_long["job_num"].astype(int)
df79_occ_long["year"] = df79_occ_long["year"].astype(int)
df79_occ_long = df79_occ_long.drop(columns=["variable"]).sort_values(["CASEID", "year", "job_num"]).reset_index(drop=True)

# Education
educ_cols = [c for c in df79.columns if c.startswith("HGCREV")]
educ_long = df79[["CASEID"] + educ_cols].melt(id_vars="CASEID",var_name="variable",value_name="HGCREV")
educ_long['year'] = educ_long['variable'].str.extract(r"(\d{4})$")[0].astype(int)
educ_long = educ_long.drop(columns='variable').sort_values(['CASEID','year'])
educ_long['HGCREV'] = educ_long.groupby("CASEID")['HGCREV'].ffill()

# Sample Weight
weight_cols = [c for c in df79.columns if c.startswith("SAMP")]
weight_long = df79[["CASEID"] + weight_cols].melt(id_vars="CASEID",var_name="variable",value_name="SampleWeight")
weight_long['year'] = weight_long['variable'].str.extract(r"(\d{4})$")[0].astype(int)
weight_long["SampleWeight"] = weight_long["SampleWeight"] / 100
weight_long = weight_long.drop(columns='variable')

# Age, Sex, Race, ASVAB
person_vars = df79[["CASEID", "sex", "race", "ASVAB",'Q1-3_A~Y_1979']].drop_duplicates(subset="CASEID")
person_vars["Q1-3_A~Y_1979"] = person_vars['Q1-3_A~Y_1979'] + 1900
person_vars['race'] = person_vars['race'].map(race_mapping)
person_vars['sex'] = person_vars['sex'].map(sex_mapping)

# Merge
df79_long = reduce(lambda left, right: pd.merge(left,right,on=["CASEID", "year", "job_num"],how="left"),[df79_exp,df79_hrsweek_long,df79_wage_long,df79_occ_long])
job_start79 = df79_exp.groupby(["CASEID", "job_num"], as_index=False)["year"].min().rename(columns={"year": "start_year"})
job_educ79 = job_start79.merge(educ_long[["CASEID", "year", "HGCREV"]].rename(columns={"year": "start_year", "HGCREV": "Educ"}),on=["CASEID", "start_year"],how="left",validate="m:1")
df79_long = df79_long.merge(job_educ79,on=["CASEID", "job_num"],how='left',validate='m:1')
df79_long = df79_long.merge(weight_long,on=["CASEID", "year"],how='left',validate='m:1')
df79_long = df79_long.merge(person_vars,on='CASEID',how='left',validate='m:1')
df79_long['Age'] = df79_long['year'] - df79_long['Q1-3_A~Y_1979']
df79_long = df79_long.drop(columns=['Q1-3_A~Y_1979','start_year'])

# Generating experience from tenure
last = df79_long.sort_values("year").groupby(["CASEID", "job_num"])["tenure"].last()
prior_jobs = last.groupby(level=0).cumsum() - last
df79_long["experience"] = df79_long["tenure"] + pd.MultiIndex.from_frame(df79_long[["CASEID", "job_num"]]).map(prior_jobs)
df79_long['experience'] = df79_long['experience'] / 52
df79_long['tenure'] = df79_long['tenure'] / 52
# df79_long = df79_long.dropna().reset_index(drop=True)
columns79 = ["HOURSWEEK","WAGE","OCC","Educ","SampleWeight","ASVAB","sex","race","Age","tenure","experience"]
df79_long, journey79 = dropna_with_counts(df79_long,columns79)

print("NLSY79 sample journey")
display(journey79)

NLSY79 sample journey


,Column checked,Rows before,Rows dropped,Rows remaining
0,HOURSWEEK,340320,17111,323209
1,WAGE,323209,38562,284647
2,OCC,284647,20904,263743
3,Educ,263743,5020,258723
4,SampleWeight,258723,0,258723
5,ASVAB,258723,0,258723
6,sex,258723,0,258723
7,race,258723,1923,256800
8,Age,256800,0,256800
9,tenure,256800,0,256800


* Export to csv file

In [17]:
df79_long.to_csv(os.path.join('cleaned', 'nlsy79_cleaned.csv'), index=False)

### NLSY97

In [18]:
def rename_columns_97(col):
    if 'PUBID' in col:
        return 'PUBID'
    elif 'YEMP' in col:
        clean_name = col.replace('YEMP_', '')
        if '~' in clean_name:
            clean_name = clean_name.replace('~','_')
        elif 'OCCODE-2002' in clean_name:
            clean_name = clean_name.replace('OCCODE-2002','Occupation')
        return clean_name
    elif 'EMP_' in col:
        clean_name = col.replace('EMP_','')
        if '_WEEK_' in clean_name:
            clean_name = clean_name.replace('WEEK_','')
        if 'GAP' in clean_name:
            match_gap = re.match(r'GAP_(.+)_(\d{4})\.(\d+)\.(\d+)$',clean_name)
            if match_gap:
                var, year, job_num, gap_num = match_gap.groups()
                return f"GAP_{var}.{gap_num}_{year}.{job_num}"
        return clean_name
    elif 'CV_' in col:
        clean_name = col.replace('CV_','')
        if 'WKSWK_JOB_DLI' in clean_name:
            clean_name = clean_name.replace('WKSWK_JOB_DLI','Tenure')
        elif 'MAINJOB_FLG' in clean_name:
            clean_name = clean_name.replace('MAINJOB_FLG','MainJob')
        elif 'HRLY_COMPENSATION' in clean_name:
            clean_name = clean_name.replace('HRLY_COMPENSATION','Wage')
        elif 'HRS_PER_WEEK' in clean_name:
            clean_name = clean_name.replace('HRS_PER_WEEK','Hours')

        return clean_name
    elif 'YSCH-' in col:
        if '5000' in col:
            return 'Educ_1997'
        elif '3112' in col:
            return f'Educ_{col[-4:]}'
    elif 'SAMPLING_' in col:
        return f'SampleWeight_{col[-4:]}'
    elif 'KEY!' in col:
        clean_name = col.replace('KEY!','')
        return clean_name
    return col

def clean_97_id(df):
    df = df.rename(columns=rename_columns_97).copy()
    month_cols = [c for c in df.columns if c.startswith("STARTDATE.") and "_M_" in c]
    for c in month_cols:
        year_col = c.replace("_M_", "_Y_", 1)
        date_col = c.replace("_M_", "_", 1)
        if year_col in df.columns:
            month = pd.to_numeric(df[c], errors="coerce")
            year = pd.to_numeric(df[year_col], errors="coerce")
            valid = month.between(1, 12) & year.gt(0)
            df[date_col] = year.mul(100).add(month).where(valid).astype("Int64")
    return df

def find_uid_and_tenure(exp_row, target_uid, year):
    if pd.isna(target_uid):
        return None, np.nan
    target_uid = int(target_uid)
    for column in df97_exp.columns:
        if (column.startswith("UID.") and column.endswith(f"_{year}")):
            uid_value = exp_row[column]
            if (pd.notna(uid_value) and int(uid_value) == target_uid):
                suffix = column.replace("UID.", "").split("_")[0]
                suffix = f"{int(suffix):02d}"
                tenure_column = f"Tenure.{suffix}_{year}"
                tenure = exp_row.get(tenure_column,np.nan)
                return suffix, tenure
    return None, np.nan

### 97 - Experience

In [19]:
df97_exp = pd.read_csv(os.path.join(data_folder, '97-tenure.csv'))
rename_dict = create_rnum_dict(os.path.join(data_folder, '97-tenure.cdb'))
df97_exp = df97_exp.rename(columns = rename_dict)
df97_exp = df97_exp.rename(columns = rename_columns_97)
df97_exp = df97_exp.replace([-1, -2, -3, -4,-5,], np.nan)
mainjob_records = []
for index, row in df97_exp.iterrows():
    pubid = row["PUBID"]
    for year in survey_year_97:
        mainjob_col = f"MainJob_{year}"
        if mainjob_col not in df97_exp.columns:
            continue
        mainjob_value = row[mainjob_col]
        if pd.isna(mainjob_value):
            continue
        suffix = f"{int(mainjob_value):02d}"
        uid_col = f"UID.{suffix}_{year}"
        tenure_col = f"Tenure.{suffix}_{year}"
        if uid_col not in df97_exp.columns:
            continue
        if tenure_col not in df97_exp.columns:
            continue
        uid = row[uid_col]
        tenure = row[tenure_col]
        if pd.isna(uid):
            continue
        mainjob_records.append({
            "PUBID": pubid,
            "year": int(year),
            "uid": int(uid),
            "tenure": tenure,
            "suffix": suffix
        })
df97_mainjob = pd.DataFrame(mainjob_records).sort_values(["PUBID", "year"]).reset_index(drop=True)

In [20]:
exp_row_lookup = {}
for index, row in df97_exp.iterrows():
    if pd.notna(row["PUBID"]):
        exp_row_lookup[int(row["PUBID"])] = row

mainjob_original = df97_mainjob.sort_values(["PUBID", "year"]).reset_index(drop=True).copy()
additional_rows = []
previous_mainjob = {}

for index, current in mainjob_original.iterrows():
    pubid = int(current["PUBID"])
    if pubid not in previous_mainjob:
        previous_mainjob[pubid] = current.to_dict()
        continue
    previous = previous_mainjob[pubid]
    previous_year = int(previous["year"])
    current_year = int(current["year"])
    previous_uid = int(previous["uid"])
    current_uid = int(current["uid"])

    if previous_uid != current_uid:
        exp_row = exp_row_lookup[pubid]
        old_suffix_current, old_tenure_current = find_uid_and_tenure(exp_row=exp_row,target_uid=previous_uid,year=current_year)
        if (old_suffix_current is not None and pd.notna(old_tenure_current)):
            additional_rows.append({"PUBID": pubid,"year": current_year,"uid": previous_uid,
                                    "tenure": old_tenure_current,"suffix": old_suffix_current})
        new_suffix_previous, new_tenure_previous = find_uid_and_tenure(exp_row=exp_row,target_uid=current_uid,year=previous_year)
        if new_suffix_previous is not None and pd.notna(new_tenure_previous):
            additional_rows.append({"PUBID": pubid,"year": previous_year,"uid": current_uid,
                                    "tenure": new_tenure_previous,"suffix": new_suffix_previous})
    previous_mainjob[pubid] = current.to_dict()

additional_jobs = pd.DataFrame(additional_rows,columns=["PUBID","year","uid","tenure","suffix"])
df97_mainjob = pd.concat([mainjob_original, additional_jobs],ignore_index=True)
df97_mainjob = df97_mainjob.drop_duplicates(subset=["PUBID", "year", "uid"],keep="first").sort_values(["PUBID", "year", "uid"]).reset_index(drop=True)
df97_mainjob = df97_mainjob.sort_values(["PUBID", "year"], kind="stable").reset_index(drop=True)
df97_mainjob["jobnum"] = df97_mainjob.groupby("PUBID")["uid"].transform(lambda x: x.ne(x.shift()).cumsum()).astype("Int64")

### 97 - Wage, Asvab, Occ, Highest Education, Hours per Week, Sex, Race, Sample Weight

In [21]:
df97 = pd.read_csv(os.path.join(data_folder, 'nlsy97.csv'))
rename_dict = create_rnum_dict(os.path.join(data_folder, 'nlsy97.cdb'))
df97 = df97.rename(columns = rename_dict)
df97 = df97.rename(columns = rename_columns_97)
df97 = df97.replace([-1, -2, -3, -4,-5], np.nan)
df97.drop(columns=['SAMPLE_TYPE_1997'],inplace=True)

asvab_cols = [c for c in df97.columns if c.startswith("ASVAB")]
subtests = ["GS", "AR", "WK", "PC", "NO", "CS", "MK", "MC"]
for test in subtests:
    pos = f"ASVAB_{test}_ABILITY_EST_POS"
    neg = f"ASVAB_{test}_ABILITY_EST_NEG"
    df97[f"ASVAB_{test}"] = (df97[pos]/1000).combine_first(-df97[neg]/1000)
    df97 = df97.drop(columns=[pos,neg])
asvab_cols = [c for c in df97.columns if c.startswith("ASVAB")]
col97 = [c[:-5] for c in df97.columns if c != 'PUBID']
df97["group_3month"] = (df97["BDATE_Y_1997"].astype(str) + "_Q" + (((df97["BDATE_M_1997"] - 1) // 3) + 1).astype(str))

wage_cols = [c for c in df97.columns if c.startswith('Wage')]
occ_cols = [c for c in df97.columns if c.startswith('Occupation')]

for c in wage_cols:
    year = int(c[-4:])
    df97[c] = (df97[c] / 100)/ cpi_map[year]

for c in occ_cols:
    original = df97[c].copy()
    if int(c[-4:]) < 2002:
        df97[c] = original.map(occ70_dict)
    elif int(c[-4:]) == 2002:
        df97[c] = original.map(occ00_dict)
    elif int(c[-4:]) > 2002:
        df97[c] = (original / 10).map(occ00_dict)

* ASVAB

In [22]:
# # 3 Month and One year comparison

# mean1 = (df97.groupby("group_3month")[asvab_cols].mean().sort_index())
# mean2 = (df97.groupby("BDATE_Y_1997")[asvab_cols].mean().sort_index())

# ax = mean1.plot(figsize=(12, 6),linewidth=1)

# year_positions = {}
# for position, group_label in enumerate(mean1.index):
#     year = str(group_label).split("_")[0]
#     year_positions.setdefault(year, []).append(position)

# year_positions = {year: np.mean(positions) for year, positions in year_positions.items()}

# for column, line in zip(asvab_cols, ax.lines):
#     x_dots = []
#     y_dots = []
#     for year, value in mean2[column].items():
#         year_key = str(int(year))
#         if year_key in year_positions and pd.notna(value):
#             x_dots.append(year_positions[year_key])
#             y_dots.append(value)
#     ax.scatter(x_dots, y_dots, color=line.get_color(), edgecolor="black", linewidth=0.5,
#         s=45, zorder=5, label="_nolegend_")
# q1_positions = [i for i, label in enumerate(mean1.index) if str(label).upper().endswith("_Q1")]

# q1_labels = [str(mean1.index[i]).upper() for i in q1_positions]
# ax.set_xlabel("")
# ax.set_xticks(q1_positions)
# ax.set_xticklabels(q1_labels, rotation=0)
# ax.set_ylabel("Mean ASVAB score")
# ax.set_title("Three-Month ASVAB Means compared with One-Year Mean - NLSY97 cohort")
# # ax.get_legend().remove()
# ax.get_legend()
# ax.grid(alpha=0.3)

# plt.tight_layout()
# plt.show()

In [23]:
# Normalization of ASVAB scores using Three-Year age group
std_cols = [f"{col}_std" for col in asvab_cols]
df97 = df97.copy()
df97[std_cols] = df97.groupby("group_3month")[asvab_cols].transform(lambda x: (x - x.mean()) / x.std(ddof=1))
# df_79ability[std_cols].plot.density()

# Applying PCA analysis
valid_r = df97[std_cols].notna().all(axis=1)
valid_df = df97.loc[valid_r, std_cols]
pca = PCA(n_components=2)
pca_scores = pca.fit_transform(valid_df)
print("Variance explained by PC1:", f"{pca.explained_variance_ratio_[0]:.3f}")
df97["ASVAB"] = np.nan
df97.loc[valid_r, "ASVAB"] = pca_scores[:, 0]
# df97["ASVAB"] = df97["ASVAB"].rank(pct=True)
# df97["ASVAB"] = (df97["ASVAB"] - df97["ASVAB"].mean()) / df97["ASVAB"].std(ddof=1)
df97.drop(columns=asvab_cols+std_cols+["group_3month","BDATE_M_1997"], inplace=True)
df97.dropna(subset=['ASVAB'], inplace=True)

Variance explained by PC1: 0.672


In [24]:
# Hoursweek
hour_cols = [c for c in df97.columns if c.startswith('Hours')]
df97_hrs = df97.melt(id_vars=["PUBID"],value_vars=hour_cols,var_name="variable",value_name="Hours").dropna(subset=["Hours"])
df97_hrs[['suffix','year']] = df97_hrs['variable'].str.extract(r"Hours\.(\d+)_(\d{4})")
df97_hrs['suffix'] = df97_hrs['suffix'].astype(int)
df97_hrs['year'] = df97_hrs['year'].astype(int)
df97_hrs['suffix'] = df97_hrs['suffix'].apply(lambda x:f'{int(x):02d}')
df97_hrs = df97_hrs.drop(columns=['variable']).sort_values(['PUBID','year','suffix']).reset_index(drop=True)

# Wage
df97_wage = df97.melt(id_vars=["PUBID"],value_vars=wage_cols,var_name="variable",value_name="Wage").dropna(subset=["Wage"])
df97_wage[['suffix','year']] = df97_wage['variable'].str.extract(r"Wage\.(\d+)_(\d{4})")
df97_wage['suffix'] = df97_wage['suffix'].astype(int)
df97_wage['year'] = df97_wage['year'].astype(int)
df97_wage['suffix'] = df97_wage['suffix'].apply(lambda x:f'{int(x):02d}')
df97_wage = df97_wage.drop(columns=['variable']).sort_values(['PUBID','year','suffix']).reset_index(drop=True)

# Occupation
df97_occ = df97.melt(id_vars=["PUBID"],value_vars=occ_cols,var_name="variable",value_name="Occupation").dropna(subset=["Occupation"])
df97_occ[['suffix','year']] = df97_occ['variable'].str.extract(r"Occupation\.(\d+)_(\d{4})")
df97_occ['suffix'] = df97_occ['suffix'].astype(int)
df97_occ['year'] = df97_occ['year'].astype(int)
df97_occ['suffix'] = df97_occ['suffix'].apply(lambda x:f'{int(x):02d}')
df97_occ = df97_occ.drop(columns=['variable']).sort_values(['PUBID','year','suffix']).reset_index(drop=True)

# Education 
educ_cols = [c for c in df97.columns if c.startswith('Educ')]
df97_educ = df97[['PUBID']+educ_cols].melt(id_vars='PUBID',var_name='variable',value_name='Education')
df97_educ['year'] = df97_educ['variable'].str.extract(r"(\d{4})$")[0].astype(int)
df97_educ = df97_educ.drop(columns='variable').sort_values(['PUBID','year'])
df97_educ["Education"] = df97_educ["Education"].replace(95, np.nan)
df97_educ['Education'] = df97_educ.groupby("PUBID")['Education'].ffill()

# Sample Weight 
weight_cols = [c for c in df97.columns if c.startswith('Sample')]
df97_weight = df97[['PUBID']+weight_cols].melt(id_vars='PUBID',var_name='variable',value_name='Sample Weight')
df97_weight['year'] = df97_weight['variable'].str.extract(r"(\d{4})$")[0].astype(int)
df97_weight['Sample Weight'] = df97_weight['Sample Weight'] / 100
df97_weight = df97_weight[df97_weight['Sample Weight'] != 0]
df97_weight = df97_weight.drop(columns='variable').reset_index(drop=True)

# Age, Sex, Race, ASVAB
df97_person = df97[["PUBID", "SEX_1997", "RACE_ETHNICITY_1997", "ASVAB",'BDATE_Y_1997']].drop_duplicates(subset="PUBID")
# df97_person['RACE_ETHNICITY_1997'] = df97_person['RACE_ETHNICITY_1997'].map(race_mapping)
df97_person['SEX_1997'] = df97_person['SEX_1997'].map(sex_mapping)
df97_person['RACE_ETHNICITY_1997'] = df97_person['RACE_ETHNICITY_1997'].map(race_map_97)
df97_person = df97_person.rename(columns={'SEX_1997':'Sex', 'RACE_ETHNICITY_1997':'Race'})

# Merge
df97_long = reduce(lambda left, right: pd.merge(left,right,on=["PUBID", "year", "suffix"],how="left"),[df97_mainjob,df97_hrs,df97_wage,df97_occ])
job_start = df97_mainjob.groupby(["PUBID", "jobnum"], as_index=False)["year"].min().rename(columns={"year": "start_year"})
job_educ = job_start.merge(df97_educ[["PUBID", "year", "Education"]].rename(columns={"year": "start_year", "Education": "Educ"}),on=["PUBID", "start_year"],how="left",validate="m:1")
df97_long = df97_long.merge(job_educ[["PUBID", "jobnum", "Educ"]],on=["PUBID", "jobnum"],how="left",validate="m:1")
df97_long = df97_long.merge(df97_weight,on=["PUBID", "year"],how='left',validate='m:1')
df97_long = df97_long.merge(df97_person,on='PUBID',how='left',validate='m:1')
df97_long['Age'] = df97_long['year'] - df97_long['BDATE_Y_1997']
df97_long.drop(columns=['suffix','BDATE_Y_1997'],inplace=True)

# Generate Experience
last = (df97_long.sort_values(["PUBID", "year"]).groupby(["PUBID", "jobnum"])["tenure"].last())
prior_jobs = last.groupby(level=0).cumsum() - last
df97_long["Experience"] = (df97_long["tenure"] + pd.MultiIndex.from_frame(df97_long[["PUBID", "jobnum"]]).map(prior_jobs)) / 52
df97_long['tenure'] = df97_long['tenure'] / 52
# df97_long = df97_long.dropna().reset_index(drop=True)
columns97 = ["Hours","Wage","Occupation","Educ","Sample Weight","ASVAB","Sex","Race","Age","tenure","Experience"]
df97_long, journey97 = dropna_with_counts(df97_long, columns97)

display(journey97)

,Column checked,Rows before,Rows dropped,Rows remaining
0,Hours,161638,32132,129506
1,Wage,129506,6941,122565
2,Occupation,122565,34397,88168
3,Educ,88168,0,88168
4,Sample Weight,88168,20140,68028
5,ASVAB,68028,0,68028
6,Sex,68028,0,68028
7,Race,68028,0,68028
8,Age,68028,0,68028
9,tenure,68028,657,67371


* Export to CSV file

In [25]:
df97_long.to_csv(os.path.join('cleaned', 'nlsy97_cleaned.csv'), index=False)